
# WattWise AI — Home Energy Comfort & Anomaly Intelligence

**AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026 | BharatCares**

### Capstone Project — Prathamesh Jagtap

WattWise AI analyzes 10-minute household energy observations to answer four practical questions:

1. **How much appliance energy is likely to be used in the next interval?**
2. **Which environmental and time variables are most useful for prediction?**
3. **When does the home experience high energy-load pressure?**
4. **Which observations look unusual, and when do high energy use and comfort deviations occur together?**

The project combines EDA, time-aware regression, explainability, anomaly detection and a configurable Comfort–Energy Conflict Index.


In [ ]:

# Install only the libraries used by this notebook.
%pip install -q numpy pandas matplotlib seaborn scikit-learn


In [ ]:

import io
import urllib.request
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, IsolationForest
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42
TARGET = "Appliances"

print("Environment ready.")



## 1. Data Loading

The primary source is the **UCI Machine Learning Repository Appliances Energy Prediction** dataset. UCI describes it as a multivariate time-series regression dataset with 19,735 observations collected at 10-minute intervals for about 4.5 months. It contains appliance energy use, indoor temperature/humidity, outdoor weather and lighting variables.

The notebook avoids a specialized dataset package and uses a direct CSV URL with a local-file fallback. This reduces dependency-related failures.


In [ ]:

# Robust dataset loader:
# 1) use a local CSV if supplied,
# 2) otherwise download the research repository CSV,
# 3) otherwise try the UCI legacy URL.

LOCAL_FILE = Path("energydata_complete.csv")
GITHUB_URL = "https://raw.githubusercontent.com/LuisM78/Appliances-energy-prediction-data/master/energydata_complete.csv"
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00374/energydata_complete.csv"

def load_energy_data():
    if LOCAL_FILE.exists():
        print("Loading local energydata_complete.csv")
        return pd.read_csv(LOCAL_FILE)

    urls = [GITHUB_URL, UCI_URL]
    errors = []

    for url in urls:
        try:
            print("Trying:", url)
            df = pd.read_csv(url)
            print("Download successful.")
            return df
        except Exception as exc:
            errors.append(f"{url} -> {type(exc).__name__}: {exc}")

    raise RuntimeError(
        "Dataset could not be loaded. Download energydata_complete.csv from the UCI/GitHub "
        "source and place it beside this notebook.\n\n" + "\n".join(errors)
    )

df = load_energy_data()

print("Raw shape:", df.shape)
display(df.head())


In [ ]:

# Basic structure and data-quality checks
print("Columns:", list(df.columns))
print("\nDuplicate rows:", int(df.duplicated().sum()))
print("\nMissing values:", int(df.isna().sum().sum()))

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"]).sort_values("date").drop_duplicates(subset=["date"]).reset_index(drop=True)

print("\nCleaned shape:", df.shape)
display(df.describe(include="all").T.head(12))



## 2. Feature Engineering

The target is appliance energy use in Wh for the next 10-minute interval.

We create:
- hour, minute block, day-of-week, month and weekend indicators
- cyclical hour encodings
- lagged appliance energy
- rolling historical energy statistics

The lag features use only past observations, so they are suitable for a one-step-ahead forecasting scenario.


In [ ]:

data = df.copy()
data["hour"] = data["date"].dt.hour
data["minute"] = data["date"].dt.minute
data["day_of_week"] = data["date"].dt.dayofweek
data["month"] = data["date"].dt.month
data["is_weekend"] = (data["day_of_week"] >= 5).astype(int)

# Cyclical time features
data["hour_decimal"] = data["hour"] + data["minute"] / 60.0
data["hour_sin"] = np.sin(2 * np.pi * data["hour_decimal"] / 24)
data["hour_cos"] = np.cos(2 * np.pi * data["hour_decimal"] / 24)

# Past-only target features
data["appliance_lag_1"] = data[TARGET].shift(1)
data["appliance_lag_3"] = data[TARGET].shift(3)
data["appliance_roll_mean_6"] = data[TARGET].shift(1).rolling(6).mean()
data["appliance_roll_mean_18"] = data[TARGET].shift(1).rolling(18).mean()

data = data.dropna().reset_index(drop=True)

print("Feature-engineered shape:", data.shape)
display(data[["date", TARGET, "appliance_lag_1", "appliance_lag_3",
              "appliance_roll_mean_6", "appliance_roll_mean_18"]].head())


## 3. Exploratory Data Analysis

In [ ]:

plt.figure(figsize=(12, 4))
plt.plot(data["date"], data[TARGET], linewidth=0.8)
plt.title("Appliance Energy Use Over Time")
plt.xlabel("Time")
plt.ylabel("Appliances (Wh)")
plt.tight_layout()
plt.show()

hour_profile = data.groupby("hour")[TARGET].mean()
plt.figure(figsize=(10, 4))
hour_profile.plot(kind="bar")
plt.title("Average Appliance Energy by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Mean Appliances (Wh)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 7))
corr_cols = [
    "Appliances", "lights", "T1", "RH_1", "T2", "RH_2", "T3", "RH_3",
    "T_out", "Press_mm_hg", "RH_out", "Windspeed", "Visibility", "Tdewpoint"
]
corr = data[corr_cols].corr(numeric_only=True)
sns.heatmap(corr, cmap="viridis", center=0)
plt.title("Correlation Heatmap — Energy, Comfort and Weather Variables")
plt.tight_layout()
plt.show()



## 4. Comfort–Energy Intelligence

Instead of claiming that high consumption automatically means "waste", WattWise AI creates a **configurable Comfort–Energy Conflict Index**.

For this prototype:
- preferred indoor temperature band = **20°C–24°C**
- preferred indoor relative humidity band = **30%–60%**
- energy pressure = percentile rank of appliance energy
- comfort deviation = normalized distance outside the configured bands

The resulting index is a relative analytical indicator, not a medical or building-standard certification.


In [ ]:

# Configurable comfort assumptions for analytical prototyping
TEMP_LOW, TEMP_HIGH = 20.0, 24.0
RH_LOW, RH_HIGH = 30.0, 60.0

def band_deviation(series, low, high):
    below = np.maximum(low - series, 0)
    above = np.maximum(series - high, 0)
    return (below + above) / max(high - low, 1e-9)

data["temp_deviation"] = band_deviation(data["T1"], TEMP_LOW, TEMP_HIGH)
data["rh_deviation"] = band_deviation(data["RH_1"], RH_LOW, RH_HIGH)

# Percentile-based relative energy pressure
data["energy_pressure"] = data[TARGET].rank(pct=True) * 100

comfort_raw = (data["temp_deviation"] + data["rh_deviation"]) / 2
comfort_pressure = comfort_raw.rank(pct=True) * 100

data["comfort_energy_conflict"] = 0.65 * data["energy_pressure"] + 0.35 * comfort_pressure

data["conflict_band"] = pd.cut(
    data["comfort_energy_conflict"],
    bins=[-np.inf, 25, 50, 75, np.inf],
    labels=["Low", "Moderate", "High", "Very High"]
)

print("Conflict-band distribution:")
display(
    data["conflict_band"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename("Percent")
    .to_frame()
)

display(
    data[["date", TARGET, "T1", "RH_1", "energy_pressure",
          "comfort_energy_conflict", "conflict_band"]].sort_values(
              "comfort_energy_conflict", ascending=False
          ).head(10)
)



## 5. Time-Ordered Machine Learning

The last 20% of chronological observations are held out as a future-like test period.

Models:
- Linear Regression — interpretable baseline
- Gradient Boosting — nonlinear boosting model
- Random Forest — nonlinear ensemble model

Evaluation:
- MAE
- RMSE
- R²


In [ ]:

# Select predictive variables.
feature_cols = [
    "lights",
    "T1", "RH_1", "T2", "RH_2", "T3", "RH_3", "T4", "RH_4", "T5", "RH_5",
    "T6", "RH_6", "T7", "RH_7", "T8", "RH_8", "T9", "RH_9",
    "T_out", "Press_mm_hg", "RH_out", "Windspeed", "Visibility", "Tdewpoint",
    "hour", "day_of_week", "month", "is_weekend", "hour_sin", "hour_cos",
    "appliance_lag_1", "appliance_lag_3", "appliance_roll_mean_6", "appliance_roll_mean_18"
]

X = data[feature_cols].copy()
y = data[TARGET].copy()

split = int(len(data) * 0.80)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

models = {
    "Linear Regression": LinearRegression(),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=250, learning_rate=0.04, max_depth=3, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=250, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE
    )
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)

print("MODEL COMPARISON")
display(results_df.round(3))


In [ ]:

best_name = results_df.loc[0, "Model"]
best_model = models[best_name]
best_pred = predictions[best_name]

plt.figure(figsize=(12, 4))
plt.plot(data["date"].iloc[split:], y_test.values, label="Actual", linewidth=1)
plt.plot(data["date"].iloc[split:], best_pred, label="Predicted", linewidth=1)
plt.title(f"Actual vs Predicted Appliance Energy — {best_name}")
plt.xlabel("Time")
plt.ylabel("Appliances (Wh)")
plt.legend()
plt.tight_layout()
plt.show()

residuals = y_test.values - best_pred
plt.figure(figsize=(8, 4))
plt.hist(residuals, bins=40)
plt.title("Prediction Residual Distribution")
plt.xlabel("Actual - Predicted (Wh)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()



## 6. Explainability — Permutation Importance

Permutation importance measures how much predictive performance changes when a feature is shuffled. It is used here as a model-reliance indicator, not as proof of causation.


In [ ]:

perm = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=8,
    random_state=RANDOM_STATE,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": perm.importances_mean,
    "Std": perm.importances_std
}).sort_values("Importance", ascending=False)

display(importance_df.head(12).round(4))

top_imp = importance_df.head(12).sort_values("Importance")
plt.figure(figsize=(9, 6))
plt.barh(top_imp["Feature"], top_imp["Importance"])
plt.title("Top Predictive Drivers — Permutation Importance")
plt.xlabel("Mean Importance")
plt.tight_layout()
plt.show()



## 7. Anomaly Detection

Isolation Forest is used to flag unusual combinations of energy, weather, comfort and temporal variables.

An anomaly is a **review candidate**, not proof of malfunction or waste.


In [ ]:

anomaly_cols = [
    TARGET, "lights", "T1", "RH_1", "T2", "RH_2", "T3", "RH_3",
    "T_out", "RH_out", "Windspeed", "hour"
]

iso = IsolationForest(
    n_estimators=250,
    contamination=0.03,
    random_state=RANDOM_STATE
)

data["anomaly_flag"] = iso.fit_predict(data[anomaly_cols])
data["anomaly_score"] = -iso.score_samples(data[anomaly_cols])
data["anomaly"] = data["anomaly_flag"].eq(-1)

anomaly_rate = data["anomaly"].mean() * 100

print(f"Detected anomaly rate: {anomaly_rate:.2f}%")

display(
    data.loc[data["anomaly"],
             ["date", TARGET, "lights", "T1", "RH_1", "T_out",
              "energy_pressure", "comfort_energy_conflict", "anomaly_score"]]
    .sort_values("anomaly_score", ascending=False)
    .head(15)
)



## 8. WattWise AI Analyst Brief

The following section converts the computed outputs into a concise, reproducible analyst brief. It is generated from the data and model outputs rather than manually typed.


In [ ]:

peak_hour = int(hour_profile.idxmax())
peak_hour_value = float(hour_profile.max())

high_conflict_share = (
    data["conflict_band"].isin(["High", "Very High"]).mean() * 100
)

high_energy_threshold = data["energy_pressure"].quantile(0.90)
high_energy_share = (data["energy_pressure"] >= high_energy_threshold).mean() * 100

top_driver = importance_df.iloc[0]["Feature"]
top_driver_importance = float(importance_df.iloc[0]["Importance"])

best_row = results_df.iloc[0]

print("WATTWISE AI ANALYST BRIEF")
print("-" * 60)
print(f"- Peak average appliance-energy hour: {peak_hour:02d}:00")
print(f"- Peak-hour mean appliance energy: {peak_hour_value:.2f} Wh")
print(f"- High/Very-High comfort-energy conflict observations: {high_conflict_share:.1f}%")
print(f"- Top 10% energy-pressure observations: {high_energy_share:.1f}%")
print(f"- Detected multivariate anomalies: {anomaly_rate:.1f}%")
print(f"- Best holdout model by RMSE: {best_name}")
print(f"- Best model MAE: {best_row['MAE']:.2f}")
print(f"- Best model RMSE: {best_row['RMSE']:.2f}")
print(f"- Best model R2: {best_row['R2']:.3f}")
print(f"- Strongest predictive driver: {top_driver} ({top_driver_importance:.3f})")
print()
print("INTERPRETATION")
print(
    "The analysis combines demand prediction, comfort-context analysis and anomaly detection. "
    "High energy use is not automatically labelled as waste; instead, the project highlights "
    "periods where energy pressure and configurable comfort deviations occur together. "
    "These outputs are intended for analytical decision support."
)



## 9. Business and SDG-Oriented Recommendations

1. **Time-aware load planning:** focus monitoring and demand-management efforts around recurring high-load periods.
2. **Comfort-aware efficiency:** inspect high conflict periods before recommending changes to heating/cooling behavior.
3. **Anomaly review:** investigate unusual energy/environment combinations for possible operational or sensor issues.
4. **Predictive monitoring:** validate the selected model on newer household/building data before deployment.
5. **Dashboard integration:** track energy pressure, conflict bands, anomalies and prediction error over time.

### SDG Alignment
- **SDG 7 — Affordable and Clean Energy:** supports energy-efficiency analysis.
- **SDG 11 — Sustainable Cities and Communities:** supports data-driven understanding of building energy behavior.
- **SDG 12 — Responsible Consumption and Production:** supports more informed resource-use decisions.



## 10. Limitations and Responsible AI

- The dataset represents one low-energy house and a historical 4.5-month observation period.
- The comfort bands used in the prototype are configurable assumptions, not universal building standards.
- High energy use does not automatically imply waste.
- Anomaly detection identifies unusual patterns; it does not diagnose faults.
- Model performance may change on other buildings, climates or appliance mixes.
- Predictive feature importance does not establish causality.
- Operational deployment should use current/local data and domain validation.


In [ ]:

# Optional: save the key outputs for inclusion in a report or dashboard.
results_df.round(4).to_csv("model_results.csv", index=False)
importance_df.round(4).to_csv("permutation_importance.csv", index=False)

summary = pd.DataFrame([{
    "peak_hour": f"{peak_hour:02d}:00",
    "peak_hour_mean_Wh": peak_hour_value,
    "high_conflict_share_pct": high_conflict_share,
    "top10_energy_pressure_share_pct": high_energy_share,
    "anomaly_rate_pct": anomaly_rate,
    "best_model": best_name,
    "best_MAE": best_row["MAE"],
    "best_RMSE": best_row["RMSE"],
    "best_R2": best_row["R2"],
    "top_driver": top_driver
}])
summary.to_csv("wattwise_summary.csv", index=False)

print("Saved:")
print("- model_results.csv")
print("- permutation_importance.csv")
print("- wattwise_summary.csv")
